# Lab 4: AI Agent Chatbot

In this lab, we build an AI Agent Chatbot that uses a trained sentiment analysis model as one of its tools. The agent follows a **perceive-reason-act** architecture and maintains conversation memory across turns.

## Objectives

- Train (or load) a sentiment analysis model
- Understand the AI Agent architecture: perceive, reason, act
- Define tools: `analyze_sentiment`, `get_help`, `summarize_conversation`
- Build a tool registry and agent loop
- Add conversation memory and error handling
- Launch a Gradio ChatInterface for interactive conversation

In [ ]:
# Run this cell in Google Colab to install dependencies
# Skip if running locally with uv
import sys
if 'google.colab' in sys.modules:
    !pip install -q keras torch torchvision gradio python-dotenv datasets transformers huggingface_hub
    print('Dependencies installed!')

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import re

print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.backend.backend()}")

## 1. Train a Sentiment Model

We train a quick sentiment model here. If you have the saved model from Lab 3 (`imdb_sentiment_model.keras`), you can load it instead.

The model uses:
- `TextVectorization` inside the model (accepts raw text)
- `Bidirectional(LSTM(64))` with `Dropout(0.5)`

In [ ]:
# Try to load saved model from Lab 3, otherwise train a new one
MODEL_PATH = "imdb_sentiment_model.keras"
LAB3_MODEL_PATH = os.path.join("..", "lab-3-text-classification", "imdb_sentiment_model.keras")

sentiment_model = None

for path in [MODEL_PATH, LAB3_MODEL_PATH]:
    if os.path.exists(path):
        try:
            sentiment_model = keras.saving.load_model(path)
            print(f"Loaded saved model from: {path}")
            break
        except Exception as e:
            print(f"Could not load model from {path}: {e}")

if sentiment_model is None:
    print("No saved model found. Training a new sentiment model...")
    print("(This will take a few minutes)\n")

    # Load and decode IMDB data
    (x_train_enc, y_train), (x_test_enc, y_test) = keras.datasets.imdb.load_data()

    word_index = keras.datasets.imdb.get_word_index()
    reverse_word_index = {value + 3: key for key, value in word_index.items()}
    reverse_word_index[0] = "<pad>"
    reverse_word_index[1] = "<start>"
    reverse_word_index[2] = "<unk>"
    reverse_word_index[3] = "<unused>"

    def decode_review(encoded_review):
        return " ".join(reverse_word_index.get(i, "?") for i in encoded_review)

    x_train_text = np.array([decode_review(seq) for seq in x_train_enc])
    x_test_text = np.array([decode_review(seq) for seq in x_test_enc])

    # Build model
    MAX_TOKENS = 10000
    MAX_LENGTH = 200
    EMBEDDING_DIM = 128

    text_vectorizer = keras.layers.TextVectorization(
        max_tokens=MAX_TOKENS,
        output_sequence_length=MAX_LENGTH,
        output_mode="int",
    )
    text_vectorizer.adapt(x_train_text)

    sentiment_model = keras.Sequential([
        keras.layers.Input(shape=(1,), dtype="string"),
        text_vectorizer,
        keras.layers.Embedding(input_dim=MAX_TOKENS, output_dim=EMBEDDING_DIM),
        keras.layers.Bidirectional(keras.layers.LSTM(64)),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(1, activation="sigmoid"),
    ])

    sentiment_model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=2, restore_best_weights=True
    )

    sentiment_model.fit(
        x_train_text, y_train,
        epochs=5,
        batch_size=64,
        validation_split=0.2,
        callbacks=[early_stopping],
        verbose=1,
    )

    test_loss, test_acc = sentiment_model.evaluate(x_test_text, y_test, verbose=0)
    print(f"\nTest accuracy: {test_acc:.4f}")

    # Save for future use
    sentiment_model.save(MODEL_PATH)
    print(f"Model saved to: {MODEL_PATH}")

In [ ]:
# Quick test of the sentiment model
test_texts = [
    "This is absolutely wonderful!",
    "Terrible and disappointing.",
]
preds = sentiment_model.predict(np.array(test_texts), verbose=0).flatten()
for text, pred in zip(test_texts, preds):
    label = "POSITIVE" if pred > 0.5 else "NEGATIVE"
    print(f"{label} ({pred:.3f}): {text}")

## 2. AI Agent Architecture

An AI Agent follows a loop:

```
User Input
    -> PERCEIVE: Parse and understand the input
    -> REASON:   Decide which tool (if any) to use
    -> ACT:      Execute the tool and format the response
    -> Return response to user
```

Key components:
- **Tool Registry**: A dictionary mapping tool names to callable functions
- **Conversation Memory**: A list of past exchanges for context
- **Parser**: Extracts intent and arguments from user input
- **Fallback Handler**: Responds gracefully when input is not understood

## 3. Define Agent Tools

In [ ]:
def analyze_sentiment(text):
    """
    Analyze the sentiment of the given text using the trained Keras model.

    Returns a dict with sentiment label and confidence score.
    """
    if not text.strip():
        return {"error": "Please provide some text to analyze."}

    pred = sentiment_model.predict(np.array([text]), verbose=0)[0][0]
    sentiment = "POSITIVE" if pred > 0.5 else "NEGATIVE"
    confidence = float(pred) if pred > 0.5 else float(1 - pred)

    return {
        "text": text,
        "sentiment": sentiment,
        "confidence": confidence,
        "raw_score": float(pred),
    }


def get_help():
    """
    Return a description of all available commands.
    """
    help_text = (
        "Available commands:\n"
        "\n"
        "  analyze <text>    - Analyze the sentiment of the given text.\n"
        "                      Example: analyze This movie was great!\n"
        "\n"
        "  help              - Show this help message.\n"
        "\n"
        "  summarize         - Summarize the conversation so far.\n"
        "\n"
        "You can also just type naturally and I will try to understand\n"
        "your intent. For example:\n"
        "  - 'What do you think about this review: ...'\n"
        "  - 'Is this positive or negative: ...'"
    )
    return help_text


def summarize_conversation(memory):
    """
    Summarize the conversation history.

    Parameters:
        memory: list of (user_msg, agent_msg) tuples
    """
    if not memory:
        return "No conversation history yet. Start by asking me to analyze some text!"

    num_exchanges = len(memory)
    sentiments_analyzed = 0
    positive_count = 0
    negative_count = 0

    for user_msg, agent_msg in memory:
        if "POSITIVE" in agent_msg:
            sentiments_analyzed += 1
            positive_count += 1
        elif "NEGATIVE" in agent_msg:
            sentiments_analyzed += 1
            negative_count += 1

    summary = (
        f"Conversation Summary\n"
        f"--------------------\n"
        f"Total exchanges: {num_exchanges}\n"
        f"Sentiments analyzed: {sentiments_analyzed}\n"
    )
    if sentiments_analyzed > 0:
        summary += (
            f"  - Positive: {positive_count}\n"
            f"  - Negative: {negative_count}\n"
        )
    return summary


# Quick test of each tool
print("=== analyze_sentiment ===")
result = analyze_sentiment("This movie was incredible!")
print(result)

print("\n=== get_help ===")
print(get_help())

print("\n=== summarize_conversation ===")
print(summarize_conversation([]))

## 4. Build the Tool Registry

In [ ]:
# Tool registry: maps tool names to their functions and descriptions
TOOL_REGISTRY = {
    "analyze": {
        "function": analyze_sentiment,
        "description": "Analyze the sentiment of text",
        "requires_arg": True,
    },
    "help": {
        "function": get_help,
        "description": "Show available commands",
        "requires_arg": False,
    },
    "summarize": {
        "function": summarize_conversation,
        "description": "Summarize the conversation history",
        "requires_arg": False,
    },
}

print("Registered tools:")
for name, info in TOOL_REGISTRY.items():
    print(f"  {name}: {info['description']}")

## 5. Implement the Agent Loop

The agent loop has three phases:

1. **Perceive**: Parse the user input to extract intent and arguments
2. **Reason**: Match the intent to a tool in the registry
3. **Act**: Execute the tool and format the response

In [ ]:
def perceive(user_input):
    """
    PERCEIVE phase: Parse user input to extract intent and arguments.

    Returns (intent, argument) tuple.
    """
    text = user_input.strip()

    if not text:
        return None, None

    text_lower = text.lower()

    # Check for explicit commands
    # Pattern: analyze "some text" or analyze some text
    analyze_match = re.match(
        r'^analyze\s+["\']?(.+?)["\']?\s*$', text, re.IGNORECASE
    )
    if analyze_match:
        return "analyze", analyze_match.group(1)

    # Check for natural language sentiment requests
    sentiment_patterns = [
        r'(?:what|how).*(?:sentiment|feel|think).*?[:\.\-]\s*(.+)',
        r'(?:is this|is it)\s+(?:positive|negative).*?[:\.\-]\s*(.+)',
        r'(?:classify|categorize|rate)\s+["\']?(.+?)["\']?\s*$',
    ]
    for pattern in sentiment_patterns:
        match = re.match(pattern, text, re.IGNORECASE)
        if match:
            return "analyze", match.group(1).strip()

    # Check for help command
    if text_lower in ("help", "commands", "what can you do", "?"):
        return "help", None

    # Check for summarize command
    if text_lower in ("summarize", "summary", "history"):
        return "summarize", None

    # Check for greeting
    greetings = ("hello", "hi", "hey", "greetings", "good morning", "good afternoon")
    if text_lower in greetings:
        return "greeting", None

    # Fallback: if it looks like text to analyze, try sentiment
    if len(text.split()) > 3:
        return "analyze", text

    return "unknown", text


# Test the perceive function
test_inputs = [
    'analyze This movie was great!',
    'What is the sentiment of: I hated this film',
    'help',
    'summarize',
    'hello',
    'xyz',
    'The acting was superb and the story was deeply moving',
]

print(f"{'Input':<55} {'Intent':<12} Argument")
print("-" * 100)
for inp in test_inputs:
    intent, arg = perceive(inp)
    arg_display = (arg[:35] + "...") if arg and len(arg) > 35 else arg
    print(f"{inp:<55} {str(intent):<12} {arg_display}")

In [ ]:
def reason_and_act(intent, argument, memory):
    """
    REASON and ACT phases: Select the appropriate tool and execute it.

    Parameters:
        intent: The parsed intent string
        argument: The argument to pass to the tool (if any)
        memory: Conversation memory list

    Returns:
        Formatted response string
    """
    # Handle greeting
    if intent == "greeting":
        return (
            "Hello! I am the Sentiment Analysis Agent. "
            "I can analyze the sentiment of text for you. "
            "Type 'help' to see available commands."
        )

    # Handle unknown intent
    if intent is None or intent == "unknown":
        return (
            "I'm not sure I understand. Here is what I can do:\n\n"
            "- Type 'analyze <text>' to analyze sentiment\n"
            "- Type 'help' for a full list of commands\n"
            "- Type 'summarize' to see our conversation summary\n"
            "- Or just paste a longer text and I will try to analyze it!"
        )

    # Look up tool in registry
    if intent in TOOL_REGISTRY:
        tool = TOOL_REGISTRY[intent]

        try:
            if intent == "analyze":
                if not argument:
                    return "Please provide some text to analyze. Example: analyze This movie was great!"
                result = tool["function"](argument)
                if "error" in result:
                    return result["error"]
                return (
                    f"Sentiment: {result['sentiment']}\n"
                    f"Confidence: {result['confidence']:.1%}\n"
                    f"Raw score: {result['raw_score']:.4f}\n"
                    f"\nText analyzed: \"{result['text']}\""
                )

            elif intent == "help":
                return tool["function"]()

            elif intent == "summarize":
                return tool["function"](memory)

        except Exception as e:
            return f"An error occurred while executing '{intent}': {str(e)}"

    return f"Tool '{intent}' not found in registry. Type 'help' for available commands."


# Test the full pipeline
test_memory = []
for inp in ["hello", "analyze This movie was fantastic!", "help", "summarize"]:
    intent, arg = perceive(inp)
    response = reason_and_act(intent, arg, test_memory)
    test_memory.append((inp, response))
    print(f"\nUser: {inp}")
    print(f"Agent: {response}")
    print("-" * 60)

## 6. Complete Agent with Conversation Memory

In [ ]:
class SentimentAgent:
    """
    AI Agent for sentiment analysis with conversation memory.

    Follows the perceive-reason-act architecture:
    1. Perceive: Parse user input to extract intent and arguments
    2. Reason: Match intent to a tool in the registry
    3. Act: Execute the tool and format the response
    """

    def __init__(self):
        self.memory = []  # List of (user_msg, agent_msg) tuples

    def chat(self, user_input):
        """
        Process a user message and return an agent response.
        """
        # Perceive
        intent, argument = perceive(user_input)

        # Reason and Act
        response = reason_and_act(intent, argument, self.memory)

        # Store in memory
        self.memory.append((user_input, response))

        return response

    def reset(self):
        """Clear conversation memory."""
        self.memory = []
        return "Conversation memory cleared."


# Create the agent
agent = SentimentAgent()

print("Agent created successfully.")
print(f"Memory size: {len(agent.memory)} exchanges")

## 7. Test Multi-Turn Conversations

In [ ]:
# Simulate a multi-turn conversation
conversation = [
    "hello",
    "analyze This movie was absolutely brilliant! The acting was superb.",
    "analyze Terrible film. Worst I have ever seen.",
    "analyze It was okay, nothing special.",
    "What is the sentiment of: I loved every moment of this beautiful story",
    "summarize",
    "help",
    "xyz",
]

agent_test = SentimentAgent()

for msg in conversation:
    response = agent_test.chat(msg)
    print(f"\nUser: {msg}")
    print(f"Agent: {response}")
    print("=" * 60)

In [ ]:
# Test error handling
edge_cases = [
    "",
    "   ",
    "analyze",
    "analyze   ",
    "a",
]

agent_edge = SentimentAgent()
print("Edge case testing:")
for msg in edge_cases:
    response = agent_edge.chat(msg)
    display_msg = repr(msg)
    print(f"\nInput: {display_msg}")
    print(f"Agent: {response}")
    print("-" * 40)

## 8. Gradio Chat Interface

Launch an interactive chatbot interface using `gr.ChatInterface`. This provides a conversational UI where you can interact with the sentiment analysis agent.

In [ ]:
import gradio as gr

# Create a fresh agent for the Gradio interface
gradio_agent = SentimentAgent()

def chat_fn(message, history):
    """
    Gradio chat function.

    Parameters:
        message: The current user message
        history: List of [user, assistant] message pairs from Gradio

    Returns:
        Agent response string
    """
    # Sync agent memory with Gradio history on first message
    # (handles page refresh)
    if len(gradio_agent.memory) == 0 and len(history) > 0:
        for user_msg, agent_msg in history:
            gradio_agent.memory.append((user_msg, agent_msg))

    response = gradio_agent.chat(message)
    return response

demo = gr.ChatInterface(
    fn=chat_fn,
    title="Sentiment Analysis Agent",
    description=(
        "Chat with the AI Agent! Try these commands:\n"
        "- **analyze** <text> -- Analyze sentiment of text\n"
        "- **help** -- Show available commands\n"
        "- **summarize** -- Summarize conversation history\n"
        "- Or just type naturally!"
    ),
    examples=[
        "hello",
        "analyze This movie was absolutely fantastic and heartwarming!",
        "analyze Terrible acting and a boring plot. Do not watch.",
        "help",
        "summarize",
    ],
    retry_btn=None,
    undo_btn=None,
)

demo.launch()